# 18. Ders: Güvenli AI Mimarisi ve İçerik Moderasyonu (Watchdog - Bölüm 2)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Bir önceki derste internetin karanlık yüzünü (toksik mesajları) tespit edebilen yapay zeka bekçimizi (Watchdog) kurduk. Ancak gerçek dünyada (Örneğin bir e-spor oyununda veya Discord sunucusunda) sadece tespit etmek yetmez, bu tespitlere göre kurallar inşa edip aksiyon almak (Moderasyon) gerekir.

Bazen bir kelime çok ağır bir küfürdür ve kullanıcı anında sistemden banlanmalıdır (Yüksek Risk). Bazen de kullanıcı sadece biraz sinirlenip argoya kaçmıştır, bu durumda yorumu tamamen silmek yerine sadece bazı kelimeleri sansürleyebiliriz (Orta Risk). Bu dersimizde yapay zeka çıktısını gelişmiş bir if-elif-else mimarisine bağlayarak kendi 'İçerik Moderasyon' (Content Moderation) filtrelerimizi yazacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Content Moderation (İçerik Moderasyonu): Sosyal platformlarda kurallara uymayan metin, resim veya videoların yapay zeka veya insan eliyle yönetilmesi.
- Auto-Censorship (Otomatik Sansür): Kötü kelimelerin yerine yıldız (***) veya maske konulması işlemi.
- False Positive (Yanlış Pozitif): Modelin, aslında temiz olan bir mesaja (Örn: "Sen deli bir dahisin!") zararlı diyerek masum kullanıcıyı haksız yere cezalandırmasıdır. Makine öğrenmesinin en tehlikeli ve en zor çözülen sorunlarından biridir.
- False Negative (Yanlış Negatif): Modelin, çok zekice/ironik bir şekilde yazılmış bir hakareti anlamayıp temiz diyerek sisteme almasıdır.

## Adım 3: Ortamı ve Bekçiyi Tekrar Hazırlamak

Toxic-BERT modelimizi tekrar `pipeline` aracılığıyla içeri alıyoruz.

In [ ]:
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# İngilizce Toksisite modelimiz
bekci_model = pipeline("text-classification", model="unitary/toxic-bert")

print("Watchdog Moderasyon Sistemi Hazır.")

## Adım 4: Gelişmiş Watchdog (Gözetmen) Fonksiyonu

Bir Discord veya Twitch sohbet botu yazdığımızı düşünelim. Kullanıcıdan gelen mesajı alacak, skoruna göre 3 farklı tepki verecek (İzin Ver, Sansürle, Sil) bir ana fonksiyon yazıyoruz.

In [ ]:
def gelismis_moderator(mesaj, kullanici_adi):
    # Modeli çalıştırıp skoru alıyoruz
    analiz = bekci_model(mesaj)[0]
    skor = analiz['score']
    
    # Adım 5: Yüksek Risk, Orta Risk, Temiz Kontrolleri
    if skor > 0.85:
        # Yüksek Risk (Ağır Küfür/Nefret)
        print(f"[SİLİNDİ] {kullanici_adi} aşırı toksik bir mesaj gönderdi. (Skor: {skor:.2f})")
        print("SİSTEM: Kullanıcı 10 dakika susturuldu (Mute).")
        
    elif skor > 0.45:
        # Orta Risk (Argo / Şüpheli)
        # Not: Gerçek hayatta burada mesajı kelime kelime sansürleyen ayrı bir kod bulunur. 
        # Biz şimdilik mesajın tamamını maskeleyeceğiz.
        print(f"[{kullanici_adi}] : *[Bu mesaj şüpheli kelimeler içerdiği için gizlenmiştir]*")
        
    else:
        # Temiz (Güvenli)
        print(f"[{kullanici_adi}] : {mesaj}")

## Adım 5: Simülasyonu Çalıştırmak

Oyun içinden gelen 3 farklı mesaj türüyle motorumuzu test edelim.

In [ ]:
print("--- OYUN SOHBETİ BAŞLADI ---\n")

gelismis_moderator("Hello guys, let's win this match!", "Player_One")

gelismis_moderator("You play terrible, what an idiot.", "Angry_Gamer")

gelismis_moderator("I will find you and kill you, you stupid piece of trash!", "Toxic_Monster")

## Adım 6: Yanlış Pozitif (False Positive) Paradoksu

Eğer güvenlik eşiğinizi çok düşük tutarsanız (Örneğin 0.30), sistem her şeye zararlı (Pozitif) diyecektir ve temiz insanların mesajlarını da haksız yere silecektir. Buna False Positive denir. Test edelim:

In [ ]:
# Masum ama içinde 'killing' (öldürmek) kelimesi geçen bir cümle. 
# Aslında "Bu şakalar beni öldürüyor" demek istiyor (Olumlu).
ironi_mesaji = "Oh man, these jokes are killing me! Haha!"

analiz = bekci_model(ironi_mesaji)[0]
print(f"İroni Cümlesinin Skoru: {analiz['score']:.4f}")

# Eğer sisteminizin eşiği çok katıysa, bu kullanıcı ceza alacaktır. 
# Bu yüzden moderasyon sistemleri (Threshold değerleri) dikkatle ayarlanmalıdır.

## Adım 7: Dene ve Keşfet - Kelime Sansürleme Algoritması

Tüm cümleyi gizlemek yerine sadece kötü kelimeyi yıldızlamak en iyi UX (Kullanıcı Deneyimi) yöntemidir. Aşağıdaki basit Python değiştirme (replace) metodunu inceleyin.

In [ ]:
mesaj = "You are an idiot and a loser!"
yasakli_kelimeler = ["idiot", "loser", "stupid"]

temizlenmis_mesaj = mesaj.lower()
for kotu_kelime in yasakli_kelimeler:
    # Kötü kelimenin uzunluğu kadar yıldız (*) oluşturuyoruz
    yildizlar = "*" * len(kotu_kelime)
    # Metin içindeki kötü kelimeyi yıldızlarla değiştiriyoruz
    temizlenmis_mesaj = temizlenmis_mesaj.replace(kotu_kelime, yildizlar)

print("Orijinal:", mesaj)
print("Sansürlü:", temizlenmis_mesaj)

## Adım 8: Sıra Sende Görevi - VIP Pozitif Yorum Ödül Sistemi

Problem: Bir platform yöneticisiniz. Platformunuzdaki kötüleri cezalandırıyorsunuz ama iyileri hiç ödüllendirmiyorsunuz! Watchdog'u tersine çevirerek sadece çok nazik mesajlar yazan kişilere "VIP" etiketi veren bir kod yazın.

Gereksinimler:
1. Adım 3'teki `bekci_model` modeli yerine, 9. derste kullandığımız varsayılan duygu analizi modelini (sentiment-analysis) koda dahil edin: `duygu_modeli = pipeline("sentiment-analysis")`
2. `odul_sistemi(mesaj, isim)` adında bir fonksiyon oluşturun.
3. Gelen mesajı modele verin ve `etiket` (label) ile `skor` (score) değerlerini çekin.
4. Eğer etiket 'POSITIVE' İSE VE skor 0.95'ten büyükse (Çok çok kibarsa):
   - Ekrana şunu yazdırın: `[VIP] {isim} : {mesaj}`
5. Eğer değilse, standart bir mesaj gibi yazdırın: `[{isim}] : {mesaj}`
6. Fonksiyonu çok nazik bir mesaj ve normal bir mesajla 2 kere çağırıp test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 18. Dersi başarıyla tamamlayarak, yapay zekanın etik (ethics) ve güvenlik (safety) boyutunu hem teorik hem de pratik (kodlayarak) anlamış oldunuz. Dünyadaki tüm büyük yapay zeka modelleri dışarıya sunulmadan önce bu filtrelerden (Watchdog sistemlerinden) geçer.

Bir sonraki dersten (Modül 5) itibaren, metinlerin ve kelimelerin dünyasından çıkıp piksellerin ve görsellerin dünyasına, Bilgisayarlı Görme (Computer Vision) evrenine giriş yapacağız.